# 06 RoPE and Position Encoding

## Problem

attention 本身为什么不知道顺序？如果 token 的内容完全一样，模型怎么区分“它在第 1 个位置”还是“它在第 20 个位置”？RoPE 又为什么会成为现代模型里非常常见的位置编码方式？

## Dependencies

建议先完成 `03_embeddings_and_language_model_inputs.ipynb`、`04_self_attention_from_scratch.ipynb`、`05_multi_head_attention.ipynb`。


## Goals

- 理解为什么 self-attention 单看内容相似性时会丢失顺序信息
- 区分绝对位置编码和相对位置编码的基本思路
- 理解 RoPE 的核心直觉：让 Q/K 随位置旋转，而不是简单相加
- 能解释为什么 RoPE 更自然地把相对位置信息带进 attention 点积

## Scope and Approach

这一节不会停在“RoPE 是一种旋转位置编码”这种口号。我们要真正回答四个问题：

1. attention 为什么天然不知道顺序。
2. 为什么简单加位置向量是一种方案，但不是唯一方案。
3. RoPE 到底对 Q/K 做了什么变换。
4. 为什么这种变换会自然影响相对位置关系。


## attention 为什么天然不知道顺序

如果你回忆上一节，attention 的核心是：

- 用 `Q @ K.T` 算相关性
- 再用这些相关性去加权汇总 `V`

问题在于，`Q` 和 `K` 的点积只关心向量内容，不天然关心“这个向量排在第几个位置”。

换句话说，如果两句话只是把 token 顺序打乱，但 token 的 embedding 集合完全一样，attention 本身并不会自动知道原始顺序。

所以位置编码回答的是一个非常根本的问题：

**模型怎么知道 token 的先后关系？**


## 从绝对位置到相对位置

最直观的做法是绝对位置编码：

- 第 0 个位置加一组位置向量
- 第 1 个位置加另一组位置向量
- 第 2 个位置再加另一组位置向量

这种方法能让模型知道“我现在在第几个位置”，但很多语言现象其实更依赖相对关系：

- 当前词离主语远不远
- 当前变量离定义点隔了几步
- 两个括号之间相距多远

RoPE 的吸引力就在这里：它不是只给你一个绝对编号，而是让位置信息更自然地进入 `Q` 和 `K` 的相似度计算里。


In [ ]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)

seq_len = 4
head_dim = 6

# 构造一个非常小的 Q/K 示例。
# 这里只是为了观察 RoPE 前后数值变化，不是完整模型输出。
Q = np.arange(seq_len * head_dim).reshape(seq_len, head_dim) * 0.1
K = Q + 0.3

print('Q before RoPE =')
print(Q)
print('K before RoPE =')
print(K)


## RoPE 到底做了什么

可以先抓住最核心的一句：

**RoPE 不是给表示直接加一个位置向量，而是让 `Q` 和 `K` 在不同位置发生不同角度的旋转。**

为什么这样做有意思？

因为注意力真正关心的是 `q_i` 和 `k_j` 的点积。如果不同位置会让向量旋转不同角度，那两个位置之间的相对差异，就会自然反映到点积结果里。

你可以把它想成：

- 同样一支箭头
- 放在不同位置时，被旋转到不同方向
- 当两个箭头做点积时，不只是内容本身，连相对角度也一起参与计算


In [ ]:
def build_rope_angles(seq_len, dim, base=10000):
    # RoPE 通常把最后一维按两两一组看待。
    # 如果 dim = 6，那么可以看成 3 组二维平面上的旋转。
    half = dim // 2

    # 每一组维度有不同的频率，低频和高频一起存在，
    # 这样模型能同时表达粗粒度和细粒度的位置变化。
    freq = 1.0 / (base ** (np.arange(half) / half))
    angles = np.outer(np.arange(seq_len), freq)
    return np.cos(angles), np.sin(angles)

def apply_rope(x):
    # x.shape = (seq_len, dim)
    cos, sin = build_rope_angles(x.shape[0], x.shape[1])

    # 偶数位和奇数位两两配对，作为一个二维平面上的坐标。
    x_even = x[:, 0::2]
    x_odd = x[:, 1::2]

    # 下面就是二维旋转公式：
    # [x', y'] = [x cos - y sin, x sin + y cos]
    rotated_even = x_even * cos - x_odd * sin
    rotated_odd = x_even * sin + x_odd * cos

    out = np.empty_like(x)
    out[:, 0::2] = rotated_even
    out[:, 1::2] = rotated_odd
    return out

Q_rope = apply_rope(Q)
K_rope = apply_rope(K)

print('Q after RoPE =')
print(Q_rope)
print('K after RoPE =')
print(K_rope)


## 为什么它会影响相对位置关系

RoPE 最妙的地方，不是“数值变漂亮了”，而是：

- 位置不同，旋转角度不同
- 旋转后再做点积，结果会受两者相对角度影响
- 相对角度本身就和相对位置有关

所以 RoPE 并不是在 attention 外面额外贴一层位置标签，而是把位置信息直接揉进了相似度计算内部。


In [ ]:
# 比较同一个 Q 和不同位置 K 的点积变化。
print('Q[0] · K[1] before =', np.dot(Q[0], K[1]))
print('Q[0] · K[1] after  =', np.dot(Q_rope[0], K_rope[1]))
print('Q[0] · K[2] before =', np.dot(Q[0], K[2]))
print('Q[0] · K[2] after  =', np.dot(Q_rope[0], K_rope[2]))


## 为什么 RoPE 常常只作用在 Q 和 K 上

这也是一个很容易糊掉的点。RoPE 主要作用在 `Q` 和 `K` 上，因为 attention 里真正决定“看谁”的，是它们的点积。

而 `V` 的任务是承载内容本身。通常我们更希望：

- `Q/K` 带着位置信息去决定相关性
- `V` 保持更像“被汇总的内容”

所以很多实现不会直接对 `V` 做同样的旋转。

## Common mistakes

- 以为 attention 已经天然知道顺序。事实上它只会比较向量，不会自动记住位置。
- 把 RoPE 只看成一个公式，而忽略它是在对 Q/K 做位置相关旋转。
- 只关注绝对位置编号，不去理解相对距离为什么对语言建模更关键。
- 以为 RoPE 是“再加一个位置向量”。它和直接相加的位置编码并不是一回事。

## Checkpoints

- 把 `seq_len` 改大到 8，观察不同位置的旋转结果会怎样变化。
- 用自己的话解释为什么 RoPE 常常只作用在 Q 和 K 上。
- 尝试解释“相对位置通过点积关系体现出来”这句话。
- 回答：RoPE 解决的是“模型知道自己在第几个位置”，还是“模型更自然地处理位置关系”？

## Summary

位置编码回答的是“这些 token 的排列顺序是什么”。RoPE 的特别之处在于，它不是把一个位置向量贴到输入上，而是让 `Q` 和 `K` 随位置旋转，从而把位置信息更自然地带进 attention 的相似度计算里。

## Next Module

下一节进入 RMSNorm 和残差连接。那是让深层 Transformer 真正稳定训练起来的重要基础设施。
